# A Practical Introduction to Linear Regression

This notebook provides a hands-on introduction to linear regression, a foundational statistical modeling technique. We will explore how to build, interpret, and visualize linear regression models using Python. Instead of using the NHANES dataset, we will work with a classic dataset containing information about tips given in a restaurant. This allows us to explore the same statistical concepts with a fresh and interesting dataset.

**What is Regression?**

Regression analysis is a set of statistical methods used to estimate the relationships between a dependent variable (often called the 'outcome') and one or more independent variables (often called 'predictors' or 'covariates'). Linear regression, specifically, models this relationship by fitting a linear equation to the observed data.

We will be using popular Python libraries for our analysis:
*   **Pandas:** For data manipulation and analysis.
*   **Numpy:** For numerical operations.
*   **Statsmodels:** For fitting and evaluating statistical models.
*   **Seaborn & Matplotlib:** For data visualization.

Let's start by importing these libraries.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.api as sm
import numpy as np

### Loading the Data

We will use the 'tips' dataset, which is conveniently available through the Seaborn library. This dataset contains records of tips given by diners, along with information about the total bill, the diner's gender, whether they were a smoker, the day of the week, the time of day, and the size of their party. We will start by loading the data and examining its structure.

In [ ]:
# Load the 'tips' dataset from Seaborn
tips = sns.load_dataset('tips')

# Display the first few rows to understand its structure
tips.head()

## Simple Linear Regression: Predicting Tips from the Total Bill

We'll begin our exploration with a simple linear regression model. Our goal is to predict the `tip` amount based on a single predictor: the `total_bill`. The `tip` is our outcome (or dependent) variable, and `total_bill` is our predictor (or independent) variable.

The model we are fitting can be expressed as:

`E[tip] = Intercept + Slope * total_bill`

This formula states that the expected tip is a linear function of the total bill. Let's use `statsmodels` to fit this model.

In [ ]:
model = sm.OLS.from_formula("tip ~ total_bill", data=tips)
result = model.fit()
result.summary()

### Interpreting the Model Parameters

Let's focus on the coefficients table in the summary above:

|             | coef   | std err | t      | P>|t|  | [0.025 | 0.975] |
|-------------|--------|---------|--------|--------|--------|--------|
| **Intercept** | 0.9203 | 0.160   | 5.761  | 0.000  | 0.606  | 1.235  |
| **total_bill**| 0.1050 | 0.007   | 14.260 | 0.000  | 0.091  | 0.119  |

- **Intercept (0.9203):** This is the predicted tip amount when the `total_bill` is zero. In this context, it doesn't have a practical meaning (a $0 bill won't get a tip), but it's a necessary part of the model that anchors the regression line.
- **total_bill coefficient (0.1050):** This is the most interesting part. It's the *slope* of our line. It means that for every additional dollar on the `total_bill`, we expect the `tip` to increase by approximately 10.5 cents. 

The p-value (`P>|t|`) for `total_bill` is 0.000, which is very small. This indicates that there is a statistically significant relationship between the total bill and the tip amount.

### Understanding R-squared

The **R-squared** value in the summary is 0.457. This statistic tells us the proportion of the variance in the outcome variable (`tip`) that is predictable from the independent variable (`total_bill`). In our case, 45.7% of the variability in tips can be explained by the total bill amount. This is a moderately strong relationship.

For a simple linear regression with one predictor, the R-squared is simply the square of the Pearson correlation coefficient between the predictor and the outcome.

In [ ]:
corr = tips[["tip", "total_bill"]].corr()
print("Squared correlation:", corr.tip.total_bill**2)

## Multiple Linear Regression: Adding More Predictors

The real power of regression comes from including multiple predictors. Let's add the `size` of the party (number of people) to our model. Does the size of the group affect the tip, even after accounting for the bill?

The new model is:
`E[tip] = Intercept + Slope_1 * total_bill + Slope_2 * size`

In [ ]:
model = sm.OLS.from_formula("tip ~ total_bill + size", data=tips)
result = model.fit()
result.summary()

### Interpreting the New Model

Now we have two coefficients to interpret:

- **total_bill (0.0927):** For every one-dollar increase in the bill, the tip is expected to increase by 9.3 cents, **holding the party size constant**.
- **size (0.1926):** For every additional person in the party, the tip is expected to increase by 19.3 cents, **holding the total bill constant**.

This concept of "holding other variables constant" is crucial in multiple regression. Each coefficient represents the unique contribution of its variable.

Notice that the coefficient for `total_bill` changed slightly (from 0.1050 to 0.0927) after we added `size` to the model. This happens because `total_bill` and `size` are correlated. When predictors are correlated, their coefficients can change when others are added or removed from the model.

The **R-squared** increased to 0.468. This means our new model explains a slightly larger portion of the variability in tips. Adding `size` has improved our model's explanatory power, although not by a large amount.

### Adding a Categorical Variable

What about categorical variables, like whether the diner is a smoker? Let's add the `smoker` variable to our model.

When we include a categorical variable, `statsmodels` automatically converts it into 'dummy variables'. If a variable has two levels (e.g., 'Yes' and 'No' for `smoker`), one level is chosen as the 'reference level' (its effect is absorbed into the intercept), and a coefficient is estimated for the other level.

In [ ]:
model = sm.OLS.from_formula("tip ~ total_bill + size + smoker", data=tips)
result = model.fit()
result.summary()

### Interpreting the Categorical Coefficient

In the output, you'll see a new coefficient: `smoker[T.No]`. `statsmodels` chose the 'Yes' category (smokers) as the reference level.

- **smoker[T.No] (-0.0633):** This means that a non-smoking party is expected to tip about 6.3 cents *less* than a smoking party, holding the total bill and party size constant. 

However, look at the p-value (`P>|t|`) for this coefficient: it's 0.613. Since this is much larger than the conventional significance level of 0.05, we conclude that there is **no statistically significant difference** in tipping between smokers and non-smokers after accounting for the bill size and party size. The R-squared value also barely changed, reinforcing that `smoker` is not a strong predictor in this model.

## Visualizing Regression Models

Visualizations are essential for understanding and diagnosing regression models. They can help us see the relationships we've modeled and check if our model's assumptions are met.

### Visualizing the Fit

A great way to visualize a multiple regression model is to plot the relationship between the outcome and one predictor while holding the other predictors at fixed values. Seaborn's `regplot` is excellent for this, but for a more direct look at our `statsmodels` results, we can use a partial regression plot.

#### Component-Plus-Residual (Partial Residual) Plots

A partial residual plot shows the relationship between the outcome and one predictor after accounting for the effects of the other predictors in the model. It helps isolate the effect of a single variable.

Let's look at the partial residual plot for `total_bill`.

In [ ]:
# This is not part of the main Statsmodels API, so needs to be imported separately
from statsmodels.graphics.regressionplots import plot_ccpr

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)
plot_ccpr(result, "total_bill", ax=ax)
ax.lines[0].set_alpha(0.3) # Make points transparent
ax.lines[1].set_color('orange') # Highlight the regression line
ax.grid(True)
plt.title('Partial Residual Plot for Total Bill')
plt.show();

The plot shows the data points scattered around a line representing the relationship between `tip` and `total_bill` after controlling for `size` and `smoker`. The upward slope confirms the positive association we saw in our model summary.

### Residual Analysis

Residuals are the errors of our model: the difference between the actual `tip` and the `tip` predicted by our model (`residual = actual - predicted`). Analyzing residuals is crucial for checking the assumptions of linear regression.

A common diagnostic plot is the **residuals vs. fitted values plot**. The fitted values are the predictions our model makes for each data point. Ideally, the residuals should be randomly scattered around zero with no clear pattern.

In [ ]:
fitted_values = result.fittedvalues
residuals = result.resid

plt.figure(figsize=(10, 6))
sns.scatterplot(x=fitted_values, y=residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values (Predicted Tip)")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residuals vs. Fitted Values")
plt.grid(True)
plt.show();

In this plot, we can see a potential issue. The scatter of the residuals appears to widen as the fitted values increase. This cone-like shape suggests **heteroscedasticity**, which means the variance of the errors is not constant. For lower predicted tips, the errors are small and clustered together. For higher predicted tips, the errors are much more spread out. This is a common pattern in financial data and indicates that our model is better at predicting smaller tips than larger ones.

While our model provides useful insights, this residual plot tells us that a more advanced model (e.g., one that transforms the `tip` variable or uses a different modeling technique) might be necessary for more accurate predictions.

## Conclusion

In this notebook, we've walked through the fundamentals of linear regression. We started with a simple model and progressively made it more complex by adding predictors. We learned how to interpret regression coefficients for both continuous and categorical variables, understand the meaning of R-squared, and use visualizations to diagnose our model.

Key takeaways:
- Linear regression models the linear relationship between predictors and an outcome.
- Coefficients represent the change in the outcome for a one-unit change in a predictor, holding other predictors constant.
- R-squared measures the proportion of variance in the outcome explained by the model.
- Visual diagnostics, especially residual plots, are critical for assessing a model's validity and identifying areas for improvement.